In [0]:
%sql
SELECT
    DISTINCT TRIM(CNTRY)
FROM workspace.bronze.erp_loc_a101_raw

#Initialisations

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, add_months, substring, to_date, expr
from pyspark.sql.types import StringType

In [0]:
RENAME_MAP = {
    "CID" : "customer_id",
    "CNTRY" : "country"
}

#Reading the bronze table in

In [0]:
df = spark.read.table("workspace.bronze.erp_loc_a101_raw")
df.show()

#Transformations

##1. Trimming all text column to remove white spaces

In [0]:
for field in df.schema:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
df.show()

##2. Normalising the abbreviations of the country column

In [0]:
df = df.withColumn(
    "CNTRY",
    F.when(F.upper(col("CNTRY")) == "US", "United States").
    when(F.upper(col("CNTRY")) == "USA", "United States").
    when(F.upper(col("CNTRY")) == "DE", "Germany").
    when(F.upper(col("CNTRY")) == "", None).
    otherwise(col("CNTRY"))
)
df.show()

##3. Preparing the key for data modelling in the gold layer.

In [0]:
df = df.withColumn(
    "customer_key",
    F.concat(substring(col("CID"), 1, 2), substring(col("CID"), 4, 9999))
)

df.show()

##4. Renaming the column names to business freindly names

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)
df.show()

#Writting to the silver layer

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.erp_customer_location")
)